# t3：读懂 model_json_schema()

用 Pydantic 生成 JSON Schema 并逐条阅读。Schema 是模型对外的"数据契约"，工具调用（function calling）、表单生成、文档都靠它。

In [1]:
import json
from pydantic import BaseModel, Field


class Publisher(BaseModel):
    name: str = Field(min_length=1)
    country: str = "unknown"


class Book(BaseModel):
    title: str = Field(min_length=1)
    author: str = Field(min_length=1)
    pages: int | None = Field(default=None, ge=1, le=10000)
    genre: str = "unknown"
    isbn: str = Field(pattern=r"^978-\d-\d{4}-\d{4}-\d$")
    publisher: Publisher | None = None

In [2]:
schema = Book.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "$defs": {
    "Publisher": {
      "properties": {
        "name": {
          "minLength": 1,
          "title": "Name",
          "type": "string"
        },
        "country": {
          "default": "unknown",
          "title": "Country",
          "type": "string"
        }
      },
      "required": [
        "name"
      ],
      "title": "Publisher",
      "type": "object"
    }
  },
  "properties": {
    "title": {
      "minLength": 1,
      "title": "Title",
      "type": "string"
    },
    "author": {
      "minLength": 1,
      "title": "Author",
      "type": "string"
    },
    "pages": {
      "anyOf": [
        {
          "maximum": 10000,
          "minimum": 1,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Pages"
    },
    "genre": {
      "default": "unknown",
      "title": "Genre",
      "type": "string"
    },
    "isbn": {
      "pattern": "^978-\\d-\\d{4}-\\d{4}-\\d$",

## 1. `type` 字段表示什么？

`type` 是被校验 JSON 值的类型标签：

- 顶层 `"type": "object"`：整个 Book 是一个 JSON 对象（键值对），它的成员由 `properties` 描述。
- 每个属性各自的 `type`：`title`/`author`/`genre`/`isbn` 是 `"string"`，`pages` 是 `"integer"`。
- 可选字段没有单一的 type，而是用 `anyOf: [{type: integer}, {type: null}]` 表示"整数或 null"。

约束也挂在属性上：`minLength`、`maximum`/`minimum`、`pattern`、`default`。

## 2. `required` 列表里有哪几个字段？

```json
"required": ["title", "author", "isbn"]
```

只有这三个。规律是 **有没有默认值**：`pages`、`genre`、`publisher` 都有 `default`，所以不 required；`title`/`author`/`isbn` 没有默认值，必须提供。

注意 `isbn` 是必填的，而 `pages` 虽然带 `None` 类型却因为是 `default=None` 反而不必填——required 看的是默认值，不是类型里有没有 `| None`。

## 3. 嵌套模型的 JSON Schema 长什么样？

嵌套模型不会内联展开，而是被抽到 `$defs` 里，字段处用 `$ref` 引用：

```json
"publisher": {
  "anyOf": [
    { "$ref": "#/$defs/Publisher" },
    { "type": "null" }
  ],
  "default": null
}
```

- `$defs.Publisher` 内是它自己的 `properties` / `required` / `type`，规则和顶层完全一样。
- `$ref: "#/$defs/Publisher"` 表示"这里引用文档内那个定义"，避免重复展开，也允许循环引用。
- 因为字段是 `Publisher | None`，所以外面还包了一层 `anyOf + null`。若写成必填的 `publisher: Publisher`，字段处会直接是 `{"$ref": "#/$defs/Publisher"}`。

In [3]:
assert schema["required"] == ["title", "author", "isbn"]
assert "Publisher" in schema["$defs"]
assert schema["properties"]["publisher"]["anyOf"][0] == {"$ref": "#/$defs/Publisher"}
print("schema 结构符合预期")

schema 结构符合预期
